In [2]:
from sklearn.tree import DecisionTreeRegressor
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import inspect
from xgboost import XGBRegressor
import mlflow
import mlflow.sklearn
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

In [3]:
train_dataset=pd.read_csv('/home/aaic/Personal_Projects/Albany-Airbnb-Price-Prediction/central_data/feature_engineering/final_train_data.csv')
train_dataset.head()

,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,...,huizen,kingdom,london,netherlands,new,ny,paris,spain,united,utrecht
0,2016,1,100,75,0,1.0,0,1,1,5.514292,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,4,100,73,1,1.0,0,1,1,5.514292,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0
2,2013,4,100,53,0,6.0,0,1,1,5.415520,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,5,100,100,1,24.0,0,1,1,5.053609,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,5,89,100,0,8.0,0,1,1,5.188696,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0,0.0


In [4]:
train_dataset.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4164 entries, 0 to 4163
Data columns (total 95 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_since                                    4164 non-null   int64  
 1   host_response_time                            4164 non-null   int64  
 2   host_response_rate                            4164 non-null   int64  
 3   host_acceptance_rate                          4164 non-null   int64  
 4   host_is_superhost                             4164 non-null   int64  
 5   host_total_listings_count                     4164 non-null   float64
 6   host_verifications                            4164 non-null   int64  
 7   host_has_profile_pic                          4164 non-null   int64  
 8   host_identity_verified                        4164 non-null   int64  
 9   neighbourhood_cleansed                        4164 non-null   f

In [5]:
test_dataset=pd.read_csv('/home/aaic/Personal_Projects/Albany-Airbnb-Price-Prediction/central_data/feature_engineering/final_test_data.csv')
test_dataset.head()

,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood_cleansed,...,huizen,kingdom,london,netherlands,new,ny,paris,spain,united,utrecht
0,2016,5,100,98,1,2.0,0,1,1,5.386632,...,0.0,0.0,0.0,1.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,2013,2,33,50,0,1.0,0,1,1,5.517529,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0
2,2012,1,100,0,0,1.0,0,1,1,5.053609,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,3,70,67,0,1.0,2,1,1,5.216495,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0
4,2019,5,100,100,1,6.0,1,1,1,5.476502,...,0.0,0.0,0.0,0.689919,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
# Data Preparation

X_train = train_dataset.drop('price', axis=1)
y_train = train_dataset['price']

X_test = test_dataset.drop('price', axis=1)
y_test = test_dataset['price']

In [7]:
mlflow.set_experiment("Algorithm Selection Experiment")  # All 3 runs will live under this experiment

# Run 1: Decision Tree 
with mlflow.start_run(run_name="Decision Tree"):
    # Define hyperparameters
    params = {"max_depth": 5, "min_samples_split": 10}
    
    # Log parameters BEFORE training
    mlflow.log_params(params)  # log_params() logs a dict at once
    
    # Train
    dt = DecisionTreeRegressor(**params)
    dt.fit(X_train, y_train)
    preds = dt.predict(X_test)
    
    # Log metrics AFTER training
    mlflow.log_metric("mean_absolute_error", mean_absolute_error(y_test, preds))
    mlflow.log_metric("mean_squared_error", mean_squared_error(y_test, preds))
    mlflow.log_metric('root_mean_squared_error', root_mean_squared_error(y_test, preds))
    
    # Save the model as an artifact
    mlflow.sklearn.log_model(dt, "decision_tree_model")

2026/04/20 09:44:40 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/20 09:44:40 INFO mlflow.store.db.utils: Updating database tables
2026/04/20 09:44:41 INFO mlflow.tracking.fluent: Experiment with name 'Algorithm Selection Experiment' does not exist. Creating a new experiment.
2026/04/20 09:44:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 09:44:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [8]:
# Run 2: Random Forest
with mlflow.start_run(run_name="Random Forest"):
    params = {"n_estimators": 60, "max_depth": 8, "min_samples_split": 5}
    
    mlflow.log_params(params)
    
    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    
    mlflow.log_metric("mean_absolute_error", mean_absolute_error(y_test, preds))
    mlflow.log_metric("mean_squared_error", mean_squared_error(y_test, preds))
    mlflow.log_metric('root_mean_squared_error', root_mean_squared_error(y_test, preds))
    
    mlflow.sklearn.log_model(rf, "random_forest_model")

2026/04/20 09:44:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 09:44:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [9]:
# Run 3: XGBoost
with mlflow.start_run(run_name="XGBoost"):
    params = {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1, "subsample": 0.8}
    
    mlflow.log_params(params)
    
    xgb = XGBRegressor(**params)
    xgb.fit(X_train, y_train)
    preds = xgb.predict(X_test)
    
    mlflow.log_metric("mean_absolute_error", mean_absolute_error(y_test, preds))
    mlflow.log_metric("mean_squared_error", mean_squared_error(y_test, preds))
    mlflow.log_metric('root_mean_squared_error', root_mean_squared_error(y_test, preds))
    
    mlflow.xgboost.log_model(xgb, "xgboost_model")  # XGBoost has its own flavor

2026/04/20 09:45:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [49]:

inspect.signature(mean_absolute_error)

<Signature (y_true, y_pred, *, sample_weight=None, multioutput='uniform_average')>

In [46]:
help(sklearn.metrics)

Help on package sklearn.metrics in sklearn:

NAME
    sklearn.metrics - Score functions, performance metrics, pairwise metrics and distance computations.

PACKAGE CONTENTS
    _base
    _classification
    _dist_metrics
    _pairwise_distances_reduction (package)
    _pairwise_fast
    _plot (package)
    _ranking
    _regression
    _scorer
    cluster (package)
    pairwise
    tests (package)

CLASSES
    builtins.object
        sklearn.metrics._dist_metrics.DistanceMetric
        sklearn.metrics._plot.confusion_matrix.ConfusionMatrixDisplay
        sklearn.metrics._plot.regression.PredictionErrorDisplay
    sklearn.utils._plotting._BinaryClassifierCurveDisplayMixin(builtins.object)
        sklearn.metrics._plot.det_curve.DetCurveDisplay
        sklearn.metrics._plot.precision_recall_curve.PrecisionRecallDisplay
        sklearn.metrics._plot.roc_curve.RocCurveDisplay

    class ConfusionMatrixDisplay(builtins.object)
     |  ConfusionMatrixDisplay(confusion_matrix, *, display_labels

In [28]:
help(xgboost)

Help on package xgboost:

NAME
    xgboost - XGBoost: eXtreme Gradient Boosting library.

DESCRIPTION
    Contributors: https://github.com/dmlc/xgboost/blob/master/CONTRIBUTORS.md

PACKAGE CONTENTS
    _data_utils
    _typing
    callback
    collective
    compat
    config
    core
    dask (package)
    data
    federated
    libpath
    objective
    plotting
    sklearn
    spark (package)
    testing (package)
    tracker
    training

CLASSES
    abc.ABC(builtins.object)
        xgboost.core.DataIter
    builtins.object
        xgboost.core.Booster
        xgboost.core.DMatrix
            xgboost.core.ExtMemQuantileDMatrix(xgboost.core.DMatrix, xgboost.core._RefMixIn)
            xgboost.core.QuantileDMatrix(xgboost.core.DMatrix, xgboost.core._RefMixIn)
        xgboost.tracker.RabitTracker
    sklearn.base.BaseEstimator(sklearn.utils._repr_html.base.ReprHTMLMixin, sklearn.utils._repr_html.base._HTMLDocumentationLinkMixin, sklearn.utils._metadata_requests._MetadataRequester)
    